Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd


query = """

WITH valid_claims AS (
    SELECT
        claim_id
    FROM gi_claims.claim_audit_trail
    WHERE to_status IN ('INITIAL_REQUEST_APPROVED', 'CLAIM_REJECTED')
    GROUP BY claim_id
    HAVING COUNT(DISTINCT to_status) = 2
)
SELECT
    c.claim_id,
    c.patient_name,
    c.hospital_name,
    c.rejection_reason_l1,
    c.claim_closure_time
FROM corp_analytics.claims_datamodel_tbl c
JOIN valid_claims v
    ON c.claim_id = v.claim_id
WHERE DATE(c.claim_closure_time) > CURRENT_DATE - INTERVAL '30' DAY
  AND (
        c.rejection_reason_l1 IS NULL
        OR LOWER(c.rejection_reason_l1) NOT IN ('liability', 'fraud')
    )
    
"""

df  = get_sql_df(query)
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1Rhd5k9UExwljVYmCjuUr3SSjIXlzLTMVLflX_WMaQT0/edit?gid=0#gid=0"
worksheet_name = "CLAIM_INITIAL_REQUEST_APPROVED_REJECTED"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)